# Fertility and development

This notebook follows the project from the research question to the results. We download World Bank data, inspect and clean it, choose a period based on coverage, explore the patterns, make charts, and try a simple prediction model. These country-level results show associations, not causes.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import requests
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'data').is_dir():
    raise FileNotFoundError('Run from the project root or notebooks folder.')
RAW_DIR = ROOT / 'data/raw'
OUT_DIR = ROOT / 'data/processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_SAVED_SNAPSHOT = True  # Set False to collect a fresh WDI snapshot.
download_manifest = []


## 1. Download the data

We use four World Bank World Development Indicators (WDI). The notebook uses the saved raw snapshot for a repeatable run; switch `USE_SAVED_SNAPSHOT` off to request each indicator one page at a time.

In [2]:
API = 'https://api.worldbank.org/v2'
INDICATORS = {
    'SP.DYN.TFRT.IN': 'fertility',
    'SH.DYN.MORT': 'mortality',
    'NY.GDP.PCAP.KD': 'gdp_pc',
    'SE.SEC.ENRR.FE': 'education',
}

### Download country information

This request gets the country names, codes, and region labels.

In [3]:
if USE_SAVED_SNAPSHOT:
    country_data = pd.read_csv(OUT_DIR / 'country_metadata.csv', keep_default_na=False)
    country_data['is_aggregate'] = country_data['is_aggregate'].astype(str).str.lower().eq('true')
    print('Using saved country metadata:', len(country_data), 'rows')
else:
    response = requests.get(API + '/country', params={'format': 'json', 'per_page': 1000}, timeout=45)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, list) or len(payload) != 2 or not isinstance(payload[1], list):
        raise ValueError('Unexpected country metadata response')
    page_info, countries = payload
    if int(page_info['pages']) != 1 or len(countries) != int(page_info['total']):
        raise ValueError('Incomplete country metadata download')
    download_manifest.append({'url': response.url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                              'rows': len(countries)})
    print('Countries downloaded:', len(countries))


Using saved country metadata: 296 rows


### Download one indicator at a time

Each indicator has its own cell below. If one request times out, retry that indicator cell. This function handles the API pages for just that one indicator.

In [4]:
def download_indicator(indicator_code):
    if USE_SAVED_SNAPSHOT:
        print('Using saved observations for', indicator_code)
        return []
    all_rows = []
    page_number = 1
    total_pages = None
    expected_total = None
    while total_pages is None or page_number <= total_pages:
        response = requests.get(
            API + '/country/all/indicator/' + indicator_code,
            params={'source': 2, 'date': '1960:2025', 'format': 'json',
                    'per_page': 10000, 'page': page_number}, timeout=45)
        response.raise_for_status()
        payload = response.json()
        if not isinstance(payload, list) or len(payload) != 2 or not isinstance(payload[1], list):
            raise ValueError(f'Unexpected WDI response: {response.url}')
        page_info, rows_on_this_page = payload
        pages, total = int(page_info['pages']), int(page_info['total'])
        if total_pages is None:
            total_pages, expected_total = pages, total
        elif (pages, total) != (total_pages, expected_total):
            raise ValueError(f'WDI pagination changed during download: {response.url}')
        all_rows.extend(rows_on_this_page)
        download_manifest.append({'url': response.url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                                  'rows': len(rows_on_this_page), 'expected_total': total})
        page_number += 1
    if len(all_rows) != expected_total:
        raise ValueError(f'Incomplete download for {indicator_code}: {len(all_rows)} of {expected_total}')
    return all_rows


### Download Fertility

In [5]:
fertility_rows = download_indicator('SP.DYN.TFRT.IN')
if not USE_SAVED_SNAPSHOT: print('Rows in new download:', len(fertility_rows))

Using saved observations for SP.DYN.TFRT.IN


### Download Under-five mortality

In [6]:
mortality_rows = download_indicator('SH.DYN.MORT')
if not USE_SAVED_SNAPSHOT: print('Rows in new download:', len(mortality_rows))

Using saved observations for SH.DYN.MORT


### Download GDP per person

In [7]:
gdp_rows = download_indicator('NY.GDP.PCAP.KD')
if not USE_SAVED_SNAPSHOT: print('Rows in new download:', len(gdp_rows))

Using saved observations for NY.GDP.PCAP.KD


### Download Female secondary enrollment

In [8]:
education_rows = download_indicator('SE.SEC.ENRR.FE')
if not USE_SAVED_SNAPSHOT: print('Rows in new download:', len(education_rows))

Using saved observations for SE.SEC.ENRR.FE


### Combine the indicator records

In [9]:
if USE_SAVED_SNAPSHOT:
    observations = None
    print('Using the saved normalized raw WDI observations.')
else:
    observations = fertility_rows + mortality_rows + gdp_rows + education_rows
    print('Total observation rows:', len(observations))


Using the saved normalized raw WDI observations.


In [10]:
if USE_SAVED_SNAPSHOT:
    raw = pd.read_csv(RAW_DIR / 'wdi_observations.csv', keep_default_na=False, low_memory=False)
else:
    raw = pd.DataFrame([{
        'iso3': row.get('countryiso3code', ''),
        'world_bank_code': row['country']['id'],
        'country': row['country']['value'],
        'year': row['date'],
        'indicator': row['indicator']['id'],
        'value': row['value'],
    } for row in observations])
    raw.to_csv(RAW_DIR / 'wdi_observations.csv', index=False)
    (OUT_DIR / 'download_manifest.json').write_text(json.dumps(download_manifest, indent=2), encoding='utf-8')
print('Raw data:', len(raw), 'rows; missing values:', int(pd.to_numeric(raw['value'], errors='coerce').isna().sum()))
display(raw.head())


Raw data: 70224 rows; missing values: 17222


,iso3,world_bank_code,country,year,indicator,value
0,AFE,ZH,Africa Eastern and Southern,2025,SP.DYN.TFRT.IN,
1,AFE,ZH,Africa Eastern and Southern,2024,SP.DYN.TFRT.IN,4.16404379125111
2,AFE,ZH,Africa Eastern and Southern,2023,SP.DYN.TFRT.IN,4.22386057727533
3,AFE,ZH,Africa Eastern and Southern,2022,SP.DYN.TFRT.IN,4.28707989151235
4,AFE,ZH,Africa Eastern and Southern,2021,SP.DYN.TFRT.IN,4.3506907649087


### Aggregate country metadata

In [11]:
if not USE_SAVED_SNAPSHOT:
    country_data = pd.DataFrame([{
        'iso3': row['id'], 'iso2': row['iso2Code'], 'country': row['name'],
        'region': row['region']['value'], 'region_id': row['region']['id'],
        'income': row['incomeLevel']['value'],
    } for row in countries])
    country_data['is_aggregate'] = country_data.region_id.eq('NA')
print('Country metadata rows:', len(country_data))
display(country_data[['iso3', 'country', 'region', 'income']].isna().sum().rename('Missing rows').to_frame())


Country metadata rows: 296


,Missing rows
iso3,0
country,0
region,0
income,0


### Convert year and value columns

In [12]:
raw['year'] = pd.to_numeric(raw.year, errors='coerce')
raw['value'] = pd.to_numeric(raw.value, errors='coerce')
print('Rows with missing ISO3:', int(raw.iso3.eq('').sum()))
print('Rows with missing year:', int(raw.year.isna().sum()))
print('Rows with missing value:', int(raw.value.isna().sum()))

Rows with missing ISO3: 1320


Rows with missing year: 0
Rows with missing value: 17222


### Match observations to countries

In [13]:
raw_countries = raw[raw['iso3'] != ''].copy()
print('Rows with a country code:', len(raw_countries))
duplicates = raw_countries.duplicated(subset=['iso3', 'year', 'indicator'])
print('Duplicate keys:', int(duplicates.sum()))
if duplicates.any():
    raise ValueError('Duplicate country/year/indicator keys need inspection before pivoting')
unmatched = sorted(set(raw_countries.iso3) - set(country_data.iso3))
if unmatched:
    raise ValueError(f'Country codes missing from metadata: {unmatched}')
country_rows = raw_countries.merge(country_data, on='iso3', validate='many_to_one')
country_rows = country_rows[~country_rows['is_aggregate']].copy()
print('Rows kept for country analysis:', len(country_rows))


Rows with a country code: 68904


Duplicate keys: 0


Rows kept for country analysis: 57288


In [14]:
print("Rows before cleaning:", len(raw))
print("Rows after cleaning:", len(country_rows))
print("Rows removed:", len(raw) - len(country_rows))

Rows before cleaning: 70224
Rows after cleaning: 57288
Rows removed: 12936


### Make the country-year table

In [15]:
# Make one row per country and year, with an indicator column for each measure
panel = country_rows.pivot(
    index=["iso3", "year"],
    columns="indicator",
    values="value"
).reset_index()
panel = panel.rename(columns={
    "SP.DYN.TFRT.IN": "fertility",
    "SH.DYN.MORT": "mortality",
    "NY.GDP.PCAP.KD": "gdp_pc",
    "SE.SEC.ENRR.FE": "education",
})

In [16]:
# Add country and year combinations, including years with no reported data
economies = country_data[country_data["is_aggregate"] == False]
all_country_years = []
for country_code in economies["iso3"]:
    for year in range(1960, 2026):
        all_country_years.append({
            "iso3": country_code,
            "year": year
        })
all_country_years = pd.DataFrame(all_country_years)

In [17]:
panel = all_country_years.merge(
    panel,
    on=["iso3", "year"],
    how="left"
)

panel = panel.merge(economies, on="iso3", how="left")

print("Country-year rows:", len(panel))
display(panel.head())

print("Missing values by indicator:")
display(panel[["fertility", "gdp_pc", "mortality", "education"]].isna().sum())

Country-year rows: 14322


,iso3,year,gdp_pc,education,mortality,fertility,iso2,country,region,region_id,income,is_aggregate
0,ABW,1960,NaN,NaN,NaN,4.567,AW,Aruba,Latin America & Caribbean,LCN,High income,False
1,ABW,1961,NaN,NaN,NaN,4.422,AW,Aruba,Latin America & Caribbean,LCN,High income,False
2,ABW,1962,NaN,NaN,NaN,4.262,AW,Aruba,Latin America & Caribbean,LCN,High income,False
3,ABW,1963,NaN,NaN,NaN,4.107,AW,Aruba,Latin America & Caribbean,LCN,High income,False
4,ABW,1964,NaN,NaN,NaN,3.940,AW,Aruba,Latin America & Caribbean,LCN,High income,False


Missing values by indicator:


fertility     249
gdp_pc       2789
mortality    2461
education    8622
dtype: int64

In [18]:
panel["core_complete"] = (
    panel["fertility"].notna()
    & panel["mortality"].notna()
    & panel["gdp_pc"].notna()
    & (panel["gdp_pc"] > 0)
)

### Save the cleaned data

In [19]:
panel.to_csv(OUT_DIR / 'country_year_panel.csv', index=False)
country_data.to_csv(OUT_DIR / 'country_metadata.csv', index=False)
audit = {
    'raw_rows': len(raw),
    'empty_iso3_rows': int(raw.iso3.eq('').sum()),
    'missing_year_rows': int(raw.year.isna().sum()),
    'missing_value_rows': int(raw.value.isna().sum()),
    'duplicate_keys': int(duplicates.sum()),
    'unmatched_country_codes': unmatched,
    'aggregate_rows': len(raw_countries) - len(country_rows),
    'country_rows': len(country_rows),
    'economies': int((~country_data.is_aggregate).sum()),
    'panel_rows': len(panel),
    'missing_panel_values': {name: int(panel[name].isna().sum()) for name in
                             ['fertility', 'gdp_pc', 'mortality', 'education']},
    'nonpositive_gdp': int(panel.gdp_pc.le(0).sum()),
    'education_above_100': int(panel.education.gt(100).sum()),
    'handling': 'No imputation or automatic outlier deletion; gross enrollment may exceed 100%.',
    'raw_csv_sha256': hashlib.sha256((RAW_DIR / 'wdi_observations.csv').read_bytes()).hexdigest(),
}
(OUT_DIR / 'cleaning_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
display(pd.DataFrame([audit]).drop(columns=['missing_panel_values', 'handling', 'raw_csv_sha256']))
display(pd.DataFrame({'missing panel values': audit['missing_panel_values']}))
print('Saved cleaned panel, country metadata, and audit in', OUT_DIR)


,raw_rows,empty_iso3_rows,missing_year_rows,missing_value_rows,duplicate_keys,unmatched_country_codes,aggregate_rows,country_rows,economies,panel_rows,nonpositive_gdp,education_above_100
0,70224,1320,0,17222,0,[],11616,57288,217,14322,0,1104


,missing panel values
fertility,249
gdp_pc,2789
mortality,2461
education,8622


Saved cleaned panel, country metadata, and audit in D:\class assignments\CIS4930\Final Project\data\processed


## 3. Inspect coverage and choose a period

Before analyzing relationships, count how many economies have all three main variables in each year. Use years with at least 80% coverage as the main window, then keep the same economies throughout it.

In [20]:
coverage = panel.groupby("year").agg(
    fertility_countries=("fertility", "count"),
    gdp_countries=("gdp_pc", "count"),
    mortality_countries=("mortality", "count"),
    education_countries=("education", "count"),
    core_complete_countries=("core_complete", "sum"),
)
display(coverage)

,fertility_countries,gdp_countries,mortality_countries,education_countries,core_complete_countries
year,,,,,
1960,216,107,131,0,86
1961,215,112,132,0,88
1962,216,112,134,0,90
1963,215,112,137,0,89
1964,216,112,140,0,92
...,...,...,...,...,...
2021,217,208,196,148,191
2022,217,208,196,140,191
2023,217,203,196,144,190


In [21]:
fig = px.line(
    coverage,
    x=coverage.index,
    y=[
        "fertility_countries",
        "gdp_countries",
        "mortality_countries",
        "education_countries",
        "core_complete_countries",
    ],
    title="Number of countries with data each year",
    labels={
        "x": "Year",
        "value": "Number of countries",
        "variable": "Data type",
    },
)
fig.show()

### Data Scope

In [22]:
total_countries = panel["iso3"].nunique()
minimum_countries = total_countries * 0.80
eligible_years = coverage[
    coverage["core_complete_countries"] >= minimum_countries
].index
start_year = eligible_years.min()
end_year = eligible_years.max()

print("Years selected:", start_year, "to", end_year)
period = panel[
    panel["year"].between(start_year, end_year)
].copy()

Years selected: 1990 to 2024


In [23]:
complete_years_by_country = period.groupby("iso3")["core_complete"].sum()
number_of_years = end_year - start_year + 1
balanced_countries = complete_years_by_country[
    complete_years_by_country == number_of_years
].index
sample = period[period["iso3"].isin(balanced_countries)].copy()
sample["log_gdp"] = np.log(sample["gdp_pc"])
print("Countries:", sample["iso3"].nunique())
print("Country-year rows:", len(sample))

Countries: 182
Country-year rows: 6370


In [24]:
education_period = panel[
    panel["year"].between(2001, 2023)
].copy()

four_indicator_data = education_period.dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
).copy()

# GDP must be positive for later comparisons using log GDP
four_indicator_data = four_indicator_data[
    four_indicator_data["gdp_pc"] > 0
].copy()

print("Country-year rows:", len(four_indicator_data))
print("Countries with at least one complete year:",
      four_indicator_data["iso3"].nunique())
print("Countries with complete data in each year:")
display(four_indicator_data.groupby("year")["iso3"].nunique())

Country-year rows: 3137
Countries with at least one complete year: 189
Countries with complete data in each year:


year
2001    130
2002    131
2003    128
2004    139
2005    141
2006    134
2007    140
2008    140
2009    137
2010    141
2011    145
2012    137
2013    130
2014    138
2015    143
2016    139
2017    133
2018    131
2019    134
2020    136
2021    141
2022    133
2023    136
Name: iso3, dtype: int64

## 4. Explore and patterns

### Compare start and end years

In [25]:
start = sample.loc[sample.year == start_year, ['iso3', 'fertility', 'gdp_pc', 'mortality']]
end = sample.loc[sample.year == end_year].copy()
change = end.merge(start, on='iso3', suffixes=('_end', '_start'))
change['fertility_change'] = change.fertility_end - change.fertility_start
change["fertility_change_pct"] = 100 * (change["fertility_end"] / change["fertility_start"] - 1)
change['gdp_change_pct'] = 100 * (change.gdp_pc_end / change.gdp_pc_start - 1)
change['mortality_change'] = change.mortality_end - change.mortality_start
change["mortality_change_pct"] = 100 * (change["mortality_end"] / change["mortality_start"] - 1)
print("Comparing", start_year, "with", end_year)
print("Countries compared:", change["iso3"].nunique())
display(change.head())

Comparing 1990 with 2024
Countries compared: 182


,iso3,year,gdp_pc_end,education,mortality_end,fertility_end,iso2,country,region,region_id,...,core_complete,log_gdp,fertility_start,gdp_pc_start,mortality_start,fertility_change,fertility_change_pct,gdp_change_pct,mortality_change,mortality_change_pct
0,AGO,2024,2798.593928,NaN,49.0,5.048,AO,Angola,Sub-Saharan Africa,SSF,...,True,7.936872,7.272,2675.052394,212.7,-2.224,-30.583058,4.618285,-163.7,-76.962858
1,ALB,2024,6549.233874,106.715736,9.2,1.341,AL,Albania,Europe & Central Asia,ECS,...,True,8.787103,3.014,1682.881087,40.6,-1.673,-55.507631,289.167953,-31.4,-77.339901
2,AND,2024,41223.992491,101.475868,2.5,1.087,AD,Andorra,Europe & Central Asia,ECS,...,True,10.626776,1.401,31143.146876,13.0,-0.314,-22.412562,32.369387,-10.5,-80.769231
3,ARE,2024,41605.413871,100.110276,4.8,1.213,AE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan",MEA,...,True,10.635986,4.506,66481.589747,15.1,-3.293,-73.080337,-37.418142,-10.3,-68.211921
4,ARG,2024,12774.350660,NaN,9.5,1.504,AR,Argentina,Latin America & Caribbean,LCN,...,True,9.455195,3.034,8115.093864,28.8,-1.530,-50.428477,57.414700,-19.3,-67.013889


### Make charts

In [26]:
#Fertility
median_fertility = sample.groupby("year")["fertility"].median().reset_index()
fig = px.line(
    median_fertility,
    x="year",
    y="fertility",
    title="Median fertility among the selected economies",
    labels={
        "year": "Year",
        "fertility": "Births per woman",
    },
)
fig.show()

In [27]:
median_gdp = sample.groupby("year")["gdp_pc"].median().reset_index()
fig = px.line(
    median_gdp,
    x="year",
    y="gdp_pc",
    title="Median GDP per person among selected economies",
    labels={"year": "Year", "gdp_pc": "Constant 2015 US$"},
)
fig.show()

In [28]:
median_mortality = sample.groupby("year")["mortality"].median().reset_index()

fig = px.line(
    median_mortality,
    x="year",
    y="mortality",
    title="Median under-five mortality among selected economies",
    labels={"year": "Year", "mortality": "Deaths per 1,000 live births"},
)
fig.show()

In [29]:
education_trend = sample.groupby("year")["education"].agg(
    median_enrollment="median",
    countries_with_data="count",
).reset_index()
display(education_trend[["year", "countries_with_data"]])
fig = px.line(
    education_trend,
    x="year",
    y="median_enrollment",
    title="Median female secondary enrollment among reporting economies",
    labels={"year": "Year", "median_enrollment": "Gross enrollment (%)"},
)
fig.show()

,year,countries_with_data
0,1990,68
1,1991,67
2,1992,63
3,1993,63
4,1994,61
5,1995,53
6,1996,48
7,1997,40
8,1998,52
9,1999,94


In [30]:
education_trend = four_indicator_data.groupby("year")["education"].agg(
    median_enrollment="median",
    countries_with_data="count",
).reset_index()

display(education_trend[["year", "countries_with_data"]])

fig = px.line(
    education_trend,
    x="year",
    y="median_enrollment",
    title="Female secondary enrollment, 2001–2023",
    labels={
        "year": "Year",
        "median_enrollment": "Gross enrollment (%)",
    },
)
fig.show()

,year,countries_with_data
0,2001,130
1,2002,131
2,2003,128
3,2004,139
4,2005,141
5,2006,134
6,2007,140
7,2008,140
8,2009,137
9,2010,141


In [31]:
yearly_medians = sample.groupby("year")[
    ["fertility", "gdp_pc", "mortality"]
].median()

display(yearly_medians.loc[[start_year, end_year]])

,fertility,gdp_pc,mortality
year,,,
1990,3.6160,3497.194486,46.45
2024,1.9635,6419.863409,14.55


Across the selected economies, median fertility and under-five mortality declined while median GDP per person increased between 1990 and 2024.

In [32]:
# Set each indicator's starting value to 100
trend_index = yearly_medians / yearly_medians.loc[start_year] * 100
trend_index = trend_index.rename(columns={
    "fertility": "Fertility",
    "gdp_pc": "GDP per person",
    "mortality": "Under-five mortality",
})
fig = px.line(
    trend_index,
    x=trend_index.index,
    y=["Fertility", "GDP per person", "Under-five mortality"],
    title="How the three indicators changed over time",
    labels={
        "x": "Year",
        "value": "Index (1990 = 100)",
        "variable": "Indicator",
    },
)
fig.show()

In [33]:
trend = sample.groupby("year")[["fertility", "gdp_pc"]].median()
trend = trend / trend.loc[start_year] * 100
trend = trend.rename(columns={
    "fertility": "Fertility",
    "gdp_pc": "GDP per person",
})

fig = px.line(
    trend,
    x=trend.index,
    y=["Fertility", "GDP per person"],
    title="Fertility and GDP per person over time",
    labels={"x": "Year", "value": "Index (1990 = 100)", "variable": "Indicator"},
)
fig.show()

In [34]:
chosen_year = 2010
chosen_data = sample.loc[sample.year == chosen_year].copy()

In [35]:
fig = px.scatter(
    chosen_data,
    x="gdp_pc",
    y="fertility",
    hover_name="country",
    log_x=True,
    title="Fertility and GDP per person in " + str(chosen_year),
    labels={
        "gdp_pc": "GDP per person (constant 2015 US$, log scale)",
        "fertility": "Births per woman",
    },
    trendline="ols",
    trendline_options={"log_x": True},
)
fig.show()

In [36]:
fig = px.scatter(
    end,
    x="gdp_pc",
    y="fertility",
    hover_name="country",
    log_x=True,
    title="Fertility and GDP per person in " + str(end_year),
    labels={
        "gdp_pc": "GDP per person (constant 2015 US$, log scale)",
        "fertility": "Births per woman",
    },
    trendline="ols",
    trendline_options={"log_x": True},
)
fig.show()

In [37]:
trend = sample.groupby("year")[["fertility", "mortality"]].median()
trend = trend / trend.loc[start_year] * 100

trend = trend.rename(columns={
    "fertility": "Fertility",
    "mortality": "Under-five mortality",
})

fig = px.line(
    trend,
    x=trend.index,
    y=["Fertility", "Under-five mortality"],
    title="Fertility and under-five mortality over time",
    labels={"x": "Year", "value": "Index (1990 = 100)", "variable": "Indicator"},
)
fig.show()

In [38]:
fig = px.scatter(
    chosen_data,
    x="mortality",
    y="fertility",
    hover_name="country",
    title="Fertility and under-five mortality in " + str(chosen_year),
    labels={
        "mortality": "Under-five deaths per 1,000 live births",
        "fertility": "Births per woman",
    },
    trendline="ols",
)
fig.show()

In [39]:
fig = px.scatter(
    end,
    x="mortality",
    y="fertility",
    hover_name="country",
    title="Fertility and under-five mortality in " + str(end_year),
    labels={
        "mortality": "Under-five deaths per 1,000 live births",
        "fertility": "Births per woman",
    },
    trendline="ols",
)
fig.show()

In genera, economies with lower under-five mortality generally had lower fertility. This is an association across economies, not evidence that improved child survival caused fertility to fall.

In [40]:
trend = four_indicator_data.groupby("year")[
    ["fertility", "education"]
].median()

trend = trend / trend.loc[2001] * 100

trend = trend.rename(columns={
    "fertility": "Fertility",
    "education": "Female secondary enrollment",
})

fig = px.line(
    trend,
    x=trend.index,
    y=["Fertility", "Female secondary enrollment"],
    title="Fertility and female secondary enrollment over time",
    labels={"x": "Year", "value": "Index (2001 = 100)", "variable": "Indicator"},
)
fig.show()

In [41]:
education_snapshot = four_indicator_data[
    four_indicator_data["year"] == chosen_year
].copy()

print("Countries in this chart:", education_snapshot["iso3"].nunique())

fig = px.scatter(
    education_snapshot,
    x="education",
    y="fertility",
    hover_name="country",
    title="Fertility and female secondary enrollment in " + str(chosen_year),
    labels={
        "education": "Gross female secondary enrollment (%)",
        "fertility": "Births per woman",
    },
    trendline="ols",
)
fig.show()

Countries in this chart: 141


In [42]:
end_education_year = 2015
education_snapshot = four_indicator_data[
    four_indicator_data["year"] == end_education_year
].copy()

print("Countries in this chart:", education_snapshot["iso3"].nunique())

fig = px.scatter(
    education_snapshot,
    x="education",
    y="fertility",
    hover_name="country",
    title="Fertility and female secondary enrollment in " + str(end_education_year),
    labels={
        "education": "Gross female secondary enrollment (%)",
        "fertility": "Births per woman",
    },
    trendline="ols",
)
fig.show()

Countries in this chart: 143


In [43]:
fig = px.scatter(
    change,
    x="gdp_change_pct",
    y="fertility_change",
    hover_name="country",
    title="Income growth and fertility change, "
          + str(start_year) + "–" + str(end_year),
    labels={
        "gdp_change_pct": "GDP per person change (%)",
        "fertility_change": "Change in births per woman",
    },
)
fig.show()

In [44]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [45]:
change["mortality_reduction"] = (
    change["mortality_start"] - change["mortality_end"]
)

change["fertility_reduction"] = (
    change["fertility_start"] - change["fertility_end"]
)

fig = px.scatter(
    change,
    x="mortality_reduction",
    y="fertility_reduction",
    hover_name="country",
    title="Reductions in child mortality and fertility, 1990–2024",
    labels={
        "mortality_reduction": "Fewer under-five deaths per 1,000 births",
        "fertility_reduction": "Fewer births per woman",
    },
    trendline="ols",
)

fig.show()

From 1990 to 2024, countries with larger reductions in under-five mortality generally also had larger reductions in fertility. The points vary considerably, so this is an overall association rather than a rule for every country. 

In [46]:
education_start = panel[panel["year"] == 2001].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
)[["iso3", "fertility", "education"]]

education_end = panel[panel["year"] == 2023].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
)[["iso3", "country", "fertility", "education"]]
education_change = education_end.merge(
    education_start,
    on="iso3",
    suffixes=("_end", "_start"),
)
education_change["education_increase"] = (
    education_change["education_end"] - education_change["education_start"]
)
education_change["fertility_reduction"] = (
    education_change["fertility_start"] - education_change["fertility_end"]
)
print("Countries compared:", len(education_change))
fig = px.scatter(
    education_change,
    x="education_increase",
    y="fertility_reduction",
    hover_name="country",
    title="Female secondary enrollment and fertility change, 2001–2023",
    labels={
        "education_increase": "Increase in gross enrollment (percentage points)",
        "fertility_reduction": "Fewer births per woman",
    },
)
fig.show()

Countries compared: 101


In [47]:
print("Spearman correlations with fertility reduction:")
print("GDP growth (1990–2024):",
      round(change["gdp_change_pct"].corr(
          change["fertility_reduction"], method="spearman"
      ), 3),
      "| Countries:", len(change))

print("Child mortality reduction (1990–2024):",
      round(change["mortality_reduction"].corr(
          change["fertility_reduction"], method="spearman"
      ), 3),
      "| Countries:", len(change))

print("Female secondary enrollment increase (2001–2023):",
      round(education_change["education_increase"].corr(
          education_change["fertility_reduction"], method="spearman"
      ), 3),
      "| Countries:", len(education_change))

Spearman correlations with fertility reduction:
GDP growth (1990–2024): -0.031 | Countries: 182
Child mortality reduction (1990–2024): 0.662 | Countries: 182
Female secondary enrollment increase (2001–2023): 0.454 | Countries: 101


In [48]:
change["fertility_reduction_pct"] = -change["fertility_change_pct"]

print(change["gdp_change_pct"].corr(
    change["fertility_reduction_pct"],
    method="spearman"
))

0.19419824808829428

From 1990 to 2024, countries with larger reductions in child mortality generally had larger reductions in fertility (Spearman = 0.662). GDP per person growth had almost no relationship with the number of births per woman reduced (−0.031), but a weak positive relationship with the percentage reduction in fertility (0.194). From 2001 to 2023, increases in female secondary enrollment had a moderate positive relationship with fertility reduction (0.454). The education result uses a different period and group of countries, so these values are not directly comparable. These relationships do not establish cause and effect.

Across countries, fertility generally fell as child mortality fell, and larger declines in the two measures often occurred together. Increases in female secondary enrollment also tended to accompany fertility declines, though that comparison covers fewer countries and a shorter period. GDP per person rose over time, but countries with faster GDP growth did not consistently have larger fertility declines.

### Calculate correlations

Correlation describes how two variables move together. It does not show that one causes the other.

In [49]:
yearly_correlations = []

for year in range(start_year, end_year + 1):
    this_year = sample[sample["year"] == year]
    row = {
        "year": year,
        "GDP per person": this_year["fertility"].corr(
            this_year["gdp_pc"], method="spearman"
        ),
        "Child mortality": this_year["fertility"].corr(
            this_year["mortality"], method="spearman"
        ),
    }
    if 2001 <= year <= 2023:
        education_year = panel[panel["year"] == year].dropna(
            subset=["fertility", "education"]
        )
        row["Female secondary enrollment"] = education_year["fertility"].corr(
            education_year["education"], method="spearman"
        )
    yearly_correlations.append(row)
yearly_correlations = pd.DataFrame(yearly_correlations)
display(yearly_correlations.round(2))

,year,GDP per person,Child mortality,Female secondary enrollment
0,1990,-0.75,0.89,NaN
1,1991,-0.75,0.88,NaN
2,1992,-0.73,0.87,NaN
3,1993,-0.72,0.86,NaN
4,1994,-0.71,0.86,NaN
5,1995,-0.71,0.86,NaN
6,1996,-0.70,0.85,NaN
7,1997,-0.70,0.85,NaN
8,1998,-0.70,0.85,NaN
9,1999,-0.69,0.84,NaN


In [50]:
fig = px.line(
    yearly_correlations,
    x="year",
    y=["GDP per person", "Child mortality", "Female secondary enrollment"],
    title="How fertility's relationships with the other indicators vary by year",
    labels={"year": "Year", "value": "Spearman correlation", "variable": "Indicator"},
)
fig.show()

Across the years studied, the country-level GDP–fertility and child mortality–fertility patterns remained fairly consistent. Higher female secondary enrollment was also generally associated with lower fertility during the years with education data. These comparisons describe differences among countries in each year, not causes of fertility change.

### Countries explorations

In [51]:
similar_countries = end[
    end["gdp_pc"].between(20000, 50000)
    & end["mortality"].between(2, 10)
].sort_values("fertility")

display(similar_countries[
    ["iso3", "country", "gdp_pc", "mortality", "fertility", "education"]
])

,iso3,country,gdp_pc,mortality,fertility,education
6994,KOR,"Korea, Rep.",37048.426400,2.8,0.748,95.891319
8578,MLT,Malta,34501.789273,5.3,1.010,NaN
328,AND,Andorra,41223.992491,2.5,1.087,101.475868
3958,ESP,Spain,29245.219245,3.2,1.100,120.838081
6532,JPN,Japan,37976.637179,2.4,1.150,NaN
6334,ITA,Italy,34508.123519,2.7,1.180,NaN
4024,EST,Estonia,20078.139762,2.0,1.180,111.402634
394,ARE,United Arab Emirates,41605.413871,4.8,1.213,100.110276
5080,GRC,Greece,21494.685606,3.8,1.240,NaN
2176,CAN,Canada,45061.386459,5.4,1.250,NaN


In [52]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [53]:
from sklearn.neighbors import NearestNeighbors

peer_data = panel[panel["year"] == 2023].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
).copy()

peer_data = peer_data[peer_data["gdp_pc"] > 0].reset_index(drop=True)

# Log GDP because incomes span a very large range.
peer_data["log_gdp"] = np.log(peer_data["gdp_pc"])

# Put the three measures on comparable scales.
peer_data["scaled_gdp"] = (
    (peer_data["log_gdp"] - peer_data["log_gdp"].mean())
    / peer_data["log_gdp"].std()
)
peer_data["scaled_mortality"] = (
    (peer_data["mortality"] - peer_data["mortality"].mean())
    / peer_data["mortality"].std()
)
peer_data["scaled_education"] = (
    (peer_data["education"] - peer_data["education"].mean())
    / peer_data["education"].std()
)

features = peer_data[
    ["scaled_gdp", "scaled_mortality", "scaled_education"]
]

print("Countries available:", len(peer_data))

Countries available: 136


In [54]:
finder = NearestNeighbors(n_neighbors=6)
finder.fit(features)

distances, neighbor_ids = finder.kneighbors(features)

results = []

for i in range(len(peer_data)):
    # The country itself is among its nearest neighbors. Leave it out.
    peer_ids = [j for j in neighbor_ids[i] if j != i][:5]
    peer_distances = [
        distance
        for distance, j in zip(distances[i], neighbor_ids[i])
        if j != i
    ][:5]

    peer_fertility = peer_data.iloc[peer_ids]["fertility"]

    peer_mean = peer_fertility.mean()
    peer_median = peer_fertility.median()

    mean_gap = peer_data.loc[i, "fertility"] - peer_mean
    median_gap = peer_data.loc[i, "fertility"] - peer_median

    results.append({
        "iso3": peer_data.loc[i, "iso3"],
        "country": peer_data.loc[i, "country"],
        "fertility": peer_data.loc[i, "fertility"],
        "peer_mean_fertility": peer_mean,
        "peer_median_fertility": peer_median,
        "mean_gap": mean_gap,
        "median_gap": median_gap,
        "absolute_gap": abs(median_gap),
        "average_peer_distance": np.mean(peer_distances),
    })

results = pd.DataFrame(results).sort_values(
    "absolute_gap", ascending=False
)

display(results.head(10))

,iso3,country,fertility,peer_mean_fertility,peer_median_fertility,mean_gap,median_gap,absolute_gap,average_peer_distance
95,NRU,Naoero,3.331,1.45900,1.439000,1.87200,1.892000,1.892000,0.202404
57,ISR,Israel,2.850,1.23840,1.200000,1.61160,1.650000,1.650000,0.090815
134,VUT,Vanuatu,3.596,2.21180,1.980000,1.38420,1.616000,1.616000,1.120488
62,KAZ,Kazakhstan,3.012,1.87380,1.439000,1.13820,1.573000,1.573000,0.283232
129,UKR,Ukraine,0.977,2.41800,2.547000,-1.44100,-1.570000,1.570000,0.532574
1,AGO,Angola,5.124,3.45300,3.605000,1.67100,1.519000,1.519000,0.700932
82,MLI,Mali,5.614,4.22908,4.192401,1.38492,1.421599,1.421599,0.801070
0,AFG,Afghanistan,4.840,3.89308,3.698000,0.94692,1.142000,1.142000,0.945039
90,NGA,Nigeria,4.482,5.15440,5.614000,-0.67240,-1.132000,1.132000,1.782617
23,BTN,Bhutan,1.462,2.50880,2.547000,-1.04680,-1.085000,1.085000,0.200834


In [55]:
finder = NearestNeighbors(n_neighbors=5)
finder.fit(features)

distances, neighbor_ids = finder.kneighbors(features)

results = []

for i in range(len(peer_data)):
    # The country itself is among its nearest neighbors. Leave it out.
    peer_ids = [j for j in neighbor_ids[i] if j != i][:5]
    peer_distances = [
        distance
        for distance, j in zip(distances[i], neighbor_ids[i])
        if j != i
    ][:5]

    peer_fertility = peer_data.iloc[peer_ids]["fertility"]

    peer_mean = peer_fertility.mean()
    peer_median = peer_fertility.median()

    mean_gap = peer_data.loc[i, "fertility"] - peer_mean
    median_gap = peer_data.loc[i, "fertility"] - peer_median

    results.append({
        "iso3": peer_data.loc[i, "iso3"],
        "country": peer_data.loc[i, "country"],
        "fertility": peer_data.loc[i, "fertility"],
        "peer_mean_fertility": peer_mean,
        "peer_median_fertility": peer_median,
        "mean_gap": mean_gap,
        "median_gap": median_gap,
        "absolute_gap": abs(median_gap),
        "average_peer_distance": np.mean(peer_distances),
    })

results = pd.DataFrame(results).sort_values(
    "absolute_gap", ascending=False
)

display(results.head(10))

,iso3,country,fertility,peer_mean_fertility,peer_median_fertility,mean_gap,median_gap,absolute_gap,average_peer_distance
95,NRU,Naoero,3.331,1.45225,1.393500,1.87875,1.937500,1.937500,0.195086
1,AGO,Angola,5.124,3.23575,3.352000,1.88825,1.772000,1.772000,0.656950
134,VUT,Vanuatu,3.596,2.07325,1.877500,1.52275,1.718500,1.718500,1.080091
57,ISR,Israel,2.850,1.24800,1.260000,1.60200,1.590000,1.590000,0.082529
82,MLI,Mali,5.614,4.07635,4.189201,1.53765,1.424799,1.424799,0.711173
129,UKR,Ukraine,0.977,2.36275,2.388500,-1.38575,-1.411500,1.411500,0.509555
62,KAZ,Kazakhstan,3.012,1.99750,1.624500,1.01450,1.387500,1.387500,0.259990
90,NGA,Nigeria,4.482,5.49525,5.837500,-1.01325,-1.355500,1.355500,1.714255
0,AFG,Afghanistan,4.840,3.78585,3.673000,1.05415,1.167000,1.167000,0.891367
71,LCA,St. Lucia,1.379,2.44550,2.459500,-1.06650,-1.080500,1.080500,0.375592


In [56]:
country_code = results.iloc[3]["iso3"]
i = peer_data.index[peer_data["iso3"] == country_code][0]

peer_ids = [j for j in neighbor_ids[i] if j != i][:5]
countries_to_show = [i] + peer_ids

display(
    peer_data.iloc[countries_to_show][
        ["country", "gdp_pc", "mortality", "education", "fertility"]
    ]
)

,country,gdp_pc,mortality,education,fertility
57,Israel,42131.741568,3.4,99.534149,2.850
3,Andorra,40227.233530,2.6,99.468742,1.082
9,Austria,46497.911464,3.3,99.539070,1.320
33,Germany,44368.992007,3.7,97.677887,1.390
61,Japan,37902.549220,2.4,101.637001,1.200


In [57]:
start_rows = panel[panel["year"] == 2001][
    ["iso3", "fertility", "gdp_pc", "mortality", "education"]
]

end_rows = panel[panel["year"] == 2023][
    ["iso3", "country", "fertility", "gdp_pc", "mortality", "education"]
]

changes = start_rows.merge(
    end_rows, on="iso3", suffixes=("_2001", "_2023")
).dropna().copy()

changes = changes[
    (changes["gdp_pc_2001"] > 0) & (changes["gdp_pc_2023"] > 0)
].reset_index(drop=True)

# Positive values mean GDP grew, mortality fell,
# education rose, or fertility fell.
changes["gdp_growth"] = np.log(
    changes["gdp_pc_2023"] / changes["gdp_pc_2001"]
)
changes["mortality_reduction"] = (
    changes["mortality_2001"] - changes["mortality_2023"]
)
changes["education_increase"] = (
    changes["education_2023"] - changes["education_2001"]
)
changes["fertility_reduction"] = (
    changes["fertility_2001"] - changes["fertility_2023"]
)

print("Countries with both endpoints:", len(changes))

Countries with both endpoints: 101


In [58]:
from sklearn.neighbors import NearestNeighbors

columns_to_compare = [
    "gdp_growth",
    "mortality_reduction",
    "education_increase",
]

scaled_changes = changes[columns_to_compare].copy()

for column in columns_to_compare:
    scaled_changes[column] = (
        (scaled_changes[column] - scaled_changes[column].mean())
        / scaled_changes[column].std()
    )

finder = NearestNeighbors(n_neighbors=6)
finder.fit(scaled_changes)

distances, neighbor_ids = finder.kneighbors(scaled_changes)

results = []

for i in range(len(changes)):
    peer_ids = [j for j in neighbor_ids[i] if j != i][:5]
    peer_distances = [
        distance
        for distance, j in zip(distances[i], neighbor_ids[i])
        if j != i
    ][:5]

    peer_fertility = changes.iloc[peer_ids]["fertility_reduction"]
    fertility_difference = (
        changes.loc[i, "fertility_reduction"]
        - peer_fertility.median()
    )

    results.append({
        "iso3": changes.loc[i, "iso3"],
        "country": changes.loc[i, "country"],
        "fertility_reduction": changes.loc[i, "fertility_reduction"],
        "peer_mean": peer_fertility.mean(),
        "peer_median": peer_fertility.median(),
        "difference_from_peer_median": fertility_difference,
        "absolute_difference": abs(fertility_difference),
        "average_peer_distance": np.mean(peer_distances),
    })

results = pd.DataFrame(results).sort_values(
    "absolute_difference", ascending=False
)

display(results.head(10))

,iso3,country,fertility_reduction,peer_mean,peer_median,difference_from_peer_median,absolute_difference,average_peer_distance
78,PSE,West Bank and Gaza,2.031,0.347000,0.177000,1.854000,1.854000,0.520324
74,PHL,Philippines,1.803,0.116400,0.010000,1.793000,1.793000,0.424027
0,AFG,Afghanistan,2.613,0.997322,0.935609,1.677391,1.677391,0.850136
34,GTM,Guatemala,2.191,0.662400,0.615000,1.576000,1.576000,0.364359
62,MNG,Mongolia,-0.500,1.089600,1.040000,-1.540000,1.540000,0.599751
64,MYS,Malaysia,1.257,-0.019200,-0.240000,1.497000,1.497000,0.352990
10,BLZ,Belize,1.519,0.200000,0.070000,1.449000,1.449000,0.398164
71,NPL,Nepal,1.809,0.907400,0.456000,1.353000,1.353000,1.017148
46,KGZ,Kyrgyz Republic,-0.300,0.907800,0.939000,-1.239000,1.239000,0.454914
79,QAT,Qatar,1.351,0.335600,0.240000,1.111000,1.111000,0.155895


In [59]:
country_code = results.iloc[0]["iso3"]
i = changes.index[changes["iso3"] == country_code][0]

peer_ids = [j for j in neighbor_ids[i] if j != i][:5]
group_ids = [i] + peer_ids

display(
    changes.iloc[group_ids][
        [
            "country",
            "gdp_growth",
            "mortality_reduction",
            "education_increase",
            "fertility_reduction",
        ]
    ]
)

country_codes = changes.iloc[group_ids]["iso3"].tolist()
fertility_lines = panel[
    panel["iso3"].isin(country_codes)
    & panel["year"].between(2001, 2023)
]

fig = px.line(
    fertility_lines,
    x="year",
    y="fertility",
    color="country",
    title="Fertility paths for countries with similar indicator changes",
    labels={"year": "Year", "fertility": "Births per woman"},
)
fig.show()

,country,gdp_growth,mortality_reduction,education_increase,fertility_reduction
78,West Bank and Gaza,0.330613,-12.5,11.764964,2.031
18,Cyprus,0.330093,1.6,11.560181,0.177
53,St. Lucia,0.282406,0.9,7.317513,0.746
15,Switzerland,0.205973,1.6,9.223747,0.050
31,United Kingdom,0.178418,1.8,11.774689,0.070
29,Fiji,0.365216,-6.6,22.836838,0.692


In [60]:
# Prepare the 2001 countries before the peer scan.
start = panel[panel["year"] == 2001].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
).copy()

start = start[start["gdp_pc"] > 0].copy()
start["log_gdp"] = np.log(start["gdp_pc"])

for column in ["log_gdp", "mortality", "education"]:
    start[column + "_scaled"] = (
        (start[column] - start[column].mean())
        / start[column].std()
    )

In [61]:
end_fertility = panel[panel["year"] == 2023][
    ["iso3", "fertility"]
].rename(columns={"fertility": "fertility_2023"})

scan = start.merge(end_fertility, on="iso3").dropna(
    subset=["fertility_2023"]
).copy()

scan["fertility_change"] = (
    scan["fertility_2023"] - scan["fertility"]
)

results = []

for _, country in scan.iterrows():
    peers = scan[
        (scan["region"] == country["region"])
        & (scan["iso3"] != country["iso3"])
    ].copy()

    peers["distance"] = np.sqrt(
        (peers["log_gdp_scaled"] - country["log_gdp_scaled"]) ** 2
        + (peers["mortality_scaled"] - country["mortality_scaled"]) ** 2
        + (peers["education_scaled"] - country["education_scaled"]) ** 2
    )

    peers = peers.nsmallest(5, "distance")

    if peers.empty:
        continue

    difference = (
        country["fertility_change"]
        - peers["fertility_change"].median()
    )

    results.append({
        "iso3": country["iso3"],
        "country": country["country"],
        "fertility_change": country["fertility_change"],
        "peer_median_change": peers["fertility_change"].median(),
        "difference": difference,
        "absolute_difference": abs(difference),
        "average_peer_distance": peers["distance"].mean(),
    })

results = pd.DataFrame(results).sort_values(
    "absolute_difference", ascending=False
)

display(results.head(10))

,iso3,country,fertility_change,peer_median_change,difference,absolute_difference,average_peer_distance
17,CAF,Central African Republic,0.205,-1.617,1.822,1.822,0.251794
97,PSE,West Bank and Gaza,-2.031,-0.223,-1.808,1.808,0.480538
34,ETH,Ethiopia,-2.590,-0.992,-1.598,1.598,0.682620
0,AFG,Afghanistan,-2.613,-1.264,-1.349,1.349,2.428843
21,CPV,Cabo Verde,-1.844,-0.510,-1.334,1.334,1.034062
43,GTM,Guatemala,-2.191,-0.939,-1.252,1.252,0.994502
93,PHL,Philippines,-1.803,-0.564,-1.239,1.239,0.471575
77,MOZ,Mozambique,-0.992,-2.195,1.203,1.203,0.386303
120,TUR,Turkiye,-0.882,0.270,-1.152,1.152,0.598612
7,BFA,Burkina Faso,-2.245,-1.111,-1.134,1.134,0.267420


In [62]:
start = panel[panel["year"] == 2001].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
).copy()

start = start[start["gdp_pc"] > 0].copy()
start["log_gdp"] = np.log(start["gdp_pc"])

# Scale the three measures so each contributes to "closeness".
for column in ["log_gdp", "mortality", "education"]:
    start[column + "_scaled"] = (
        (start[column] - start[column].mean())
        / start[column].std()
    )

country_code = "CAF"  # Change this to the country you want to explore.
chosen = start[start["iso3"] == country_code].copy()
country = chosen.iloc[0]

# Compare only with countries in the same region.
peers = start[
    (start["region"] == country["region"])
    & (start["iso3"] != country_code)
].copy()

peers["distance"] = np.sqrt(
    (peers["log_gdp_scaled"] - country["log_gdp_scaled"]) ** 2
    + (peers["mortality_scaled"] - country["mortality_scaled"]) ** 2
    + (peers["education_scaled"] - country["education_scaled"]) ** 2
)

peers = peers.nsmallest(5, "distance")
chosen["distance"] = 0
comparison = pd.concat([chosen, peers])

display(comparison[
    ["country", "gdp_pc", "mortality", "education", "fertility", "distance"]
])

,country,gdp_pc,mortality,education,fertility,distance
2087,Central African Republic,446.658634,162.5,7.64059,5.807,0.000000
1097,Burkina Faso,442.914123,171.9,7.95838,6.431,0.175384
10997,Rwanda,356.017641,166.5,10.71989,5.893,0.194231
4793,Guinea,628.320555,159.0,10.44265,5.834,0.256932
8885,Mozambique,317.185704,155.7,5.15360,5.755,0.277107
12449,Chad,572.806707,179.0,5.92101,7.231,0.355315


In [63]:
country_codes = comparison["iso3"].tolist()

history = panel[
    panel["iso3"].isin(country_codes)
    & panel["year"].between(2001, 2023)
].copy()

fig = px.line(
    history,
    x="year",
    y="fertility",
    color="country",
    title="Fertility trends for countries selected from 2001 conditions",
    labels={
        "year": "Year",
        "fertility": "Births per woman",
    },
)
fig.show()

In [64]:
import numpy as np
import pandas as pd
from itertools import combinations
from IPython.display import display

case_start = panel[panel["year"] == 2001][
    ["iso3", "country", "gdp_pc", "mortality", "education", "fertility"]
]

case_end = panel[panel["year"] == 2023][
    ["iso3", "gdp_pc", "mortality", "education", "fertility"]
]

case_rows = case_start.merge(
    case_end, on="iso3", suffixes=("_2001", "_2023")
).dropna()

case_rows = case_rows[
    (case_rows["gdp_pc_2001"] > 0)
    & (case_rows["gdp_pc_2023"] > 0)
].reset_index(drop=True)

# Match on development conditions at both endpoints.
# Fertility is deliberately excluded.
case_features = case_rows[
    [
        "gdp_pc_2001", "gdp_pc_2023",
        "mortality_2001", "mortality_2023",
        "education_2001", "education_2023",
    ]
].copy()

for column in ["gdp_pc_2001", "gdp_pc_2023"]:
    case_features[column] = np.log(case_features[column])

# Give the six measurements comparable scales.
case_scaled = (
    (case_features - case_features.mean()) / case_features.std()
).to_numpy()

case_distances = np.zeros((len(case_rows), len(case_rows)))

for i, j in combinations(range(len(case_rows)), 2):
    distance = np.sqrt(np.mean((case_scaled[i] - case_scaled[j]) ** 2))
    case_distances[i, j] = distance
    case_distances[j, i] = distance

print("Economies with complete endpoint data:", len(case_rows))

Economies with complete endpoint data: 101


In [65]:
group_size = 2
similarity_limit = 0.35

case_groups = {}

for i in range(len(case_rows)):
    neighbors = [
        j for j in np.argsort(case_distances[i]) if j != i
    ][:group_size - 1]

    members = tuple(sorted([i] + neighbors))

    largest_distance = max(
        case_distances[a, b]
        for a, b in combinations(members, 2)
    )

    if largest_distance > similarity_limit:
        continue

    group = case_rows.iloc[list(members)]

    # The dictionary avoids storing the same group more than once.
    case_groups[members] = {
        "countries": " / ".join(group["country"]),
        "codes": group["iso3"].tolist(),
        "largest_distance": largest_distance,
        "fertility_gap_2001": (
            group["fertility_2001"].max()
            - group["fertility_2001"].min()
        ),
        "fertility_gap_2023": (
            group["fertility_2023"].max()
            - group["fertility_2023"].min()
        ),
    }

if not case_groups:
    raise ValueError(
        "No groups passed. Try a smaller group_size or a larger similarity_limit."
    )

case_ranked = pd.DataFrame(list(case_groups.values()))

case_ranked["gap_growth"] = (
    case_ranked["fertility_gap_2023"]
    - case_ranked["fertility_gap_2001"]
)

case_ranked = case_ranked[
    (case_ranked["fertility_gap_2001"] <= 0.5) &
    (case_ranked["gap_growth"] >= 0.5)
].sort_values("gap_growth", ascending=False)
# Build a shortlist without repeating the same countries.
case_shortlist = []
used_countries = set()

for _, candidate in case_ranked.iterrows():
    if used_countries.isdisjoint(candidate["codes"]):
        case_shortlist.append(candidate.to_dict())
        used_countries.update(candidate["codes"])

    if len(case_shortlist) == 6:
        break

case_shortlist = pd.DataFrame(case_shortlist).reset_index(drop=True)

print("Candidate groups found:", len(case_ranked))
display(
    case_shortlist[
        [
            "countries",
            "largest_distance",
            "fertility_gap_2001",
            "fertility_gap_2023",
        ]
    ].round(3)
)

Candidate groups found: 3


,countries,largest_distance,fertility_gap_2001,fertility_gap_2023
0,Ecuador / Naoero,0.224,0.424,1.508
1,Bangladesh / Tajikistan,0.296,0.287,0.909
2,"Egypt, Arab Rep. / Philippines",0.203,0.259,0.834


In [66]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

case_number = 2
case_codes = case_shortlist.loc[case_number, "codes"]

case_history = panel[
    panel["iso3"].isin(case_codes)
    & panel["year"].between(2001, 2023)
].sort_values(["iso3", "year"])

case_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Fertility",
        "GDP per person",
        "Under-five mortality",
        "Female secondary enrollment",
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

case_measures = [
    ("fertility", 1, 1, "Births per woman"),
    ("gdp_pc", 1, 2, "Constant 2015 US$ · log scale"),
    ("mortality", 2, 1, "Deaths per 1,000 live births"),
    ("education", 2, 2, "Gross enrollment (%)"),
]

case_colors = ["#1a7774", "#d66a4f", "#6878b7", "#a4659b", "#b57b30"]

for index, code in enumerate(case_codes):
    country_history = case_history[case_history["iso3"] == code]
    country_name = country_history["country"].iloc[0]

    for measure, row, column, unit in case_measures:
        case_fig.add_trace(
            go.Scatter(
                x=country_history["year"],
                y=country_history[measure],
                name=country_name,
                legendgroup=code,
                showlegend=(measure == "fertility"),
                mode="lines",
                connectgaps=False,
                line={"color": case_colors[index], "width": 3},
            ),
            row=row,
            col=column,
        )

for measure, row, column, unit in case_measures:
    case_fig.update_yaxes(title_text=unit, row=row, col=column)

case_fig.update_yaxes(type="log", row=1, col=2)
case_fig.update_xaxes(title_text="Year", range=[2001, 2023])

case_fig.update_layout(
    title="Similar development conditions, different fertility?",
    template="plotly_white",
    height=700,
    hovermode="x unified",
    legend={"orientation": "h", "y": -0.15},
    margin={"b": 110},
)

case_fig.show()

# Check the actual numbers behind the similarity score.
display(
    case_rows[case_rows["iso3"].isin(case_codes)][
        [
            "country",
            "gdp_pc_2001", "gdp_pc_2023",
            "mortality_2001", "mortality_2023",
            "education_2001", "education_2023",
            "fertility_2001", "fertility_2023",
        ]
    ].round(2)
)

,country,gdp_pc_2001,gdp_pc_2023,mortality_2001,mortality_2023,education_2001,education_2023,fertility_2001,fertility_2023
25,"Egypt, Arab Rep.",2492.03,4111.31,44.8,22.9,77.10,83.05,3.46,2.75
74,Philippines,1807.71,3744.70,36.0,27.0,74.77,92.21,3.72,1.92


## 5. Try a simple prediction model

This optional analysis asks how well GDP per person and child mortality predict fertility for countries the model did not train on. The scores measure prediction, not causes.

In [67]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = sample[['log_gdp', 'mortality']]
y = sample.fertility
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups=sample.iso3))
model = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
predicted = model.predict(X.iloc[test_idx])
baseline = np.repeat(y.iloc[train_idx].mean(), len(test_idx))
print('Random forest MAE:', round(mean_absolute_error(y.iloc[test_idx], predicted), 3))
print('Mean baseline MAE:', round(mean_absolute_error(y.iloc[test_idx], baseline), 3))
fig = px.scatter(x=y.iloc[test_idx], y=predicted, title='Predictions for held-out countries',
                 labels={'x': 'Observed births per woman', 'y': 'Predicted births per woman'})
fig.show()

Random forest MAE: 0.588
Mean baseline MAE: 1.236


In [68]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Use the same rows for every comparison.
model_data = panel[panel["year"].between(2001, 2023)].dropna(
    subset=["fertility", "gdp_pc", "mortality", "education"]
).copy()

model_data = model_data[model_data["gdp_pc"] > 0].copy()
model_data["log_gdp"] = np.log(model_data["gdp_pc"])

feature_sets = {
    "GDP only": ["log_gdp"],
    "Mortality only": ["mortality"],
    "Education only": ["education"],
    "All three": ["log_gdp", "mortality", "education"],
    "Without GDP": ["mortality", "education"],
    "Without mortality": ["log_gdp", "education"],
    "Without education": ["log_gdp", "mortality"],
}

# Keep all years for a country together.
folds = list(
    GroupKFold(n_splits=5).split(
        model_data, groups=model_data["iso3"]
    )
)

scores = []

for name, columns in feature_sets.items():
    fold_errors = []

    for train_ids, test_ids in folds:
        model = RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42,
        )

        model.fit(
            model_data.iloc[train_ids][columns],
            model_data.iloc[train_ids]["fertility"],
        )

        predictions = model.predict(
            model_data.iloc[test_ids][columns]
        )

        error = mean_absolute_error(
            model_data.iloc[test_ids]["fertility"],
            predictions,
        )
        fold_errors.append(error)

    scores.append({
        "indicators": name,
        "MAE": np.mean(fold_errors),
    })

scores = pd.DataFrame(scores)
display(scores.sort_values("MAE"))

,indicators,MAE
3,All three,0.487999
4,Without GDP,0.490086
1,Mortality only,0.526852
6,Without education,0.538115
5,Without mortality,0.551105
2,Education only,0.581187
0,GDP only,0.669577


## 6. What we found and what to keep in mind

Summarize the patterns shown by the charts and correlation table. Compare cross-country levels with country changes over time; they can look different.

**Limitations:** The data are observational and country-level, so they do not establish causes. The balanced sample excludes countries with incomplete histories and is not population weighted. WDI values may be estimates or revised. Gross female enrollment is an education proxy, not adult attainment. Cite the World Bank and original data providers in the final report.

## What we found

**The broad trend.** Among 182 economies with complete fertility, GDP-per-person, and under-five mortality data throughout 1990–2024, median fertility fell from 3.62 to 1.96 births per woman. Median under-five mortality also fell, while median GDP per person rose. These are medians across the selected economies, not population-weighted world figures.

**Countries differ from one another.** In 2024, countries with higher GDP per person generally had lower fertility, while countries with higher under-five mortality generally had higher fertility. These cross-country relationships describe a pattern; they do not tell us what caused fertility to change.

**Changes over time tell a more complicated story.** From 1990 to 2024, countries with larger reductions in child mortality tended to have larger reductions in fertility (Spearman correlation: 0.662). GDP-per-person growth had almost no relationship with the *number* of births per woman reduced (Spearman correlation: -0.031). The education comparison uses a different period and smaller sample: among 101 countries with all four indicators at both endpoints in 2001 and 2023, increases in female secondary enrollment tended to accompany fertility reductions (Spearman correlation: 0.454). These comparisons are associations, not evidence of cause and effect.

**Prediction leaves differences unexplained.** On the 2001–2023 rows with all four indicators, a random forest using GDP per person, child mortality, and female secondary enrollment had an average held-out-country error of about 0.488 births per woman. Leaving out child mortality raised error to 0.551; leaving out education raised it to 0.538. Leaving out GDP changed error to 0.490 in this model. These results suggest that child mortality contributed the most predictive information in this comparison, while GDP added little once the other measures were available. Prediction errors remain meaningful for individual countries.

**Limitations.** The data are observational and country-level. The 1990–2024 sample excludes economies with incomplete histories and gives each economy equal weight. Education has less complete coverage, so its results use a different period and sample. World Bank historical values may be revised. Similarity searches identify countries close on selected measures, not countries with identical social or historical circumstances. None of these analyses establishes that an indicator caused fertility to rise or fall.